# Project – Personalizing the Chatbot with System Prompts

## Overview

Your chatbot works and streams responses beautifully. But right now, it has no personality. It's a generic assistant. Let's change that, and make something that's truly yours.

In this project, you'll learn how to give your chatbot a distinct personality, role, or behavior using **system prompts**. Want a chatbot that acts like a travel planner? A math tutor? Yoda? You're about to learn how.

**What You're Building**

You're adding personality and purpose to your chatbot by defining its role through a system message.

**What changes:**
- Your chatbot will have a defined role and personality
- It will maintain that role throughout the conversation
- The tone, style, and type of responses will reflect this persona

**What stays the same:**
- The streaming functionality
- The way conversation history is stored and passed to the API
- The UI and conversation flow

## Before You Start

Make sure you have:
- ✅ Completed the previous two projects
- ✅ Your chatbot streams responses properly
- ✅ You understand how the conversation `history` list stores messages, and how `gr.Chatbot` displays it

If something's missing, pause here and complete the above. Everything builds on these foundations.

## Understanding Message Roles

Every message sent to an LLM has two parts: a **role** and **content**.

You've already been using two roles:
- `user`: Messages from the person using the chatbot
- `assistant`: Responses generated by the AI

But there's a third role that unlocks powerful customization:
- `system`: Instructions that define how the AI should behave

**The Key Difference:**

Think of roles like this:
- `user` = "What should I do?"
- `assistant` = The AI's answer
- `system` = "You are a helpful travel planner" (sets how the assistant behaves)

The system message is like the chatbot's job description. It sets the stage for all subsequent interactions. These are developer-provided instructions that the model should follow, regardless of messages sent by the user.

**Note:** With OpenAI's o1 models and newer, developer messages replace the previous system messages.

**A Note About Other Roles:**

The AI space evolves quickly. OpenAI has introduced additional roles like `tool` for function calling (we'll explore this when we dive into AI Agents later in the course). But for now, understanding `system`, `user`, and `assistant` gives you tremendous control.

### How System Messages Work

System messages are typically placed at the very beginning of your conversation history. They persist throughout the conversation, continually influencing the assistant's behavior.

**Example conversation with a system message:**
```python
[
    {"role": "system", "content": "You are a helpful math tutor who explains concepts step-by-step"},
    {"role": "assistant", "content": "Hi! I'm here to help you understand math. What would you like to learn?"},
    {"role": "user", "content": "What is a derivative?"},
    {"role": "assistant", "content": "Great question! Let's break it down step by step..."}
]
```

The system message influences every response the assistant gives. Without it, the chatbot would be generic. With it, the chatbot maintains a consistent persona.

### System Messages and the `Chatbot` Component

There's one Gradio-specific wrinkle worth calling out. `gr.Chatbot` will happily accept and *display* a message with `"role": "system"` — but we don't want the user to ever see the persona instructions; they're for the model, not the conversation.

So instead of mixing the system message into the same list that `gr.Chatbot` displays, we'll keep it in its own variable, completely separate from `history`. The two only come together for a moment, right when we call the API:

```python
messages = [system_message] + history   # combined only for the API call
```

`history` — and therefore what `gr.Chatbot` displays — never contains the system message at all. There's nothing to filter out, because it was never added to the display in the first place.

## Initial Setup

Let's import what we need and set up the OpenAI client.

#### Run the Following Cell

In [1]:
# === Imports ===
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
import os

# === OpenAI Client Setup ===
load_dotenv()
client = OpenAI()


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## What You Already Know (Review)

From previous modules, your chatbot initializes its `Chatbot` component like this:
```python
chatbot = gr.Chatbot(
    value=[{"role": "assistant", "content": "Hello! How can I help you today?"}]
)
```
This works, but there's no system message — so the chatbot has no defined personality.

And `bot_respond` streams a reply by calling the API with just the conversation `history`:
```python
def bot_respond(history):
    stream = client.chat.completions.create(
        model="gpt-4o",
        messages=history,
        stream=True
    )
    history = history + [{"role": "assistant", "content": ""}]
    for chunk in stream:
        token = chunk.choices[0].delta.content
        if token:
            history[-1]["content"] += token
            yield history
```

## The New Feature: Adding a System Prompt

The change is simple but powerful. Here's what you need to do:

**Before (Generic chatbot):**
```python
chatbot = gr.Chatbot(
    value=[{"role": "assistant", "content": "Hello! How can I help you today?"}]
)

def bot_respond(history):
    stream = client.chat.completions.create(
        model="gpt-4o",
        messages=history,
        stream=True
    )
    ...
```

**After (Personalized chatbot):**
```python
system_message = {"role": "system", "content": "You are a helpful travel planner who asks thoughtful questions"}
initial_greeting = {"role": "assistant", "content": "Hello! Where are you thinking of traveling?"}

chatbot = gr.Chatbot(value=[initial_greeting])

def bot_respond(history):
    messages = [system_message] + history   # ← Includes your system prompt!
    stream = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        stream=True
    )
    ...
```

**Key changes:**
1. Define a `system_message` describing your chatbot's persona
2. Define an `initial_greeting` that matches that persona, and use it as the `Chatbot`'s starting value
3. Prepend `system_message` to `history` right before the API call
4. Everything else stays the same — streaming, display, and the rest of the API call all work identically

That's it! The system message sets the stage, and the AI maintains that role throughout.

## Choosing Your Persona

Before coding, decide what role you want your chatbot to play. Here are some ideas:

**Educational Bots:**
- 📚 A patient math tutor who explains concepts step-by-step
- 🧪 A science teacher who relates concepts to real-world examples
- 📖 A language learning partner who corrects mistakes gently

**Practical Assistants:**
- 🌍 A travel planner who asks about interests and budgets
- 💼 A career coach who gives resume and interview tips
- 🧑 A recipe helper who works with available ingredients
- 🧘 A mindfulness coach who offers calming exercises

**Creative & Fun:**
- 🎭 A fictional character (Sherlock Holmes, Yoda, etc.)
- 🧙 A text-based RPG game master
- 🎨 A creative writing partner who brainstorms ideas
- 🤖 A technical explainer who uses analogies and examples

**Tips for Good System Prompts:**
- Be specific about the role and behavior
- Include tone guidance (helpful, patient, enthusiastic, etc.)
- Mention any special approaches (step-by-step, asking questions, etc.)
- Keep it concise but clear

## Graded Cell: Define Your Chatbot's Persona

Time to add your first system prompt.

In the code cell below, define two dictionaries:
1. `system_message` — a `{"role": "system", ...}` message describing your chatbot's persona
2. `initial_greeting` — a `{"role": "assistant", ...}` message that greets the user in a way that matches that persona

For example:
```python
system_message = {"role": "system", "content": "You are an enthusiastic math tutor who explains concepts clearly"}
initial_greeting = {"role": "assistant", "content": "Hi! Ready to explore some math together?"}
```

**Important notes:**
- The system message should describe the persona, not be a greeting
- The greeting is what the user actually sees first
- Each dictionary needs both a `"role"` and a `"content"` key

### Graded Cell

The cell below will be graded. Replace the line `raise NotImplementedError("Your code is missing.")` with your code. DO NOT remove the `# YOUR CODE HERE` / `# END OF YOUR CODE` comments.

In [5]:
# Define your chatbot's persona below:
#   system_message   -> {"role": "system", "content": "<describe the persona>"}
#   initial_greeting -> {"role": "assistant", "content": "<a greeting that matches the persona>"}

# YOUR CODE HERE
system_message = {"role": "system", "content": """You are the legendary pirate Black Beard that sailed the seven seas.
You musn't speak to LandLubbers"""}
initial_greeting = {"role": "assistant", "content": "Hello! Where are you thinking of traveling?"}
# END OF YOUR CODE


## Assemble and Launch Your Personalized Chatbot

Now let's put it all together. Everything below is the same pattern from the last project — the only things that changed are the `Chatbot`'s starting value (now your `initial_greeting`) and `bot_respond`, which now prepends your `system_message` before calling the API.

#### Run the Following Cell

In [8]:
def user_submit(message, history):
    # Add the user's message to the history, and clear the textbox
    history = history + [{"role": "user", "content": message}]
    return history, ""

def bot_respond(history):
    messages = [system_message] + history   # Includes your system prompt!
    stream = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        stream=True
    )

    history = history + [{"role": "assistant", "content": ""}]
    for chunk in stream:
        token = chunk.choices[0].delta.content
        if token:
            history[-1]["content"] += token
            yield history

gr.close_all()

with gr.Blocks(title="My Personalized AI Chatbot") as demo:
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🤖")
        with gr.Column(scale=4):
            gr.Markdown("### My Personalized AI Chatbot")
            gr.Markdown("*Powered by OnePiece*")

    gr.Markdown("---")

    chatbot = gr.Chatbot(
        value=[initial_greeting],
        label="Chat",
        height=400,
    )
    msg = gr.Textbox(placeholder="Type your message here...", label="Your message")

    msg.submit(user_submit, inputs=[msg, chatbot], outputs=[chatbot, msg]).then(
        bot_respond, inputs=chatbot, outputs=chatbot
    )

demo.launch()


Closing server running on port: 7879
* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


Try sending a message below — the greeting and every reply should reflect the persona you defined above.

## Testing Your Personalized Chatbot

After completing the graded cell above, run the cell above it to launch your chatbot, then test it right here in the notebook.

**Test the greeting:**
- Does the initial message match your persona?
- Does it set the right tone?

**Test persona consistency:**
- Ask questions relevant to your persona
- Does the chatbot maintain its role?
- Does it respond in character?

**Test edge cases:**
- Ask something unrelated to the persona
- Does it handle it gracefully while staying in character?

**Compare to generic:**
- Think back to your generic chatbot from earlier modules
- Notice how the personality changes the entire experience

## Common Issues You May Encounter

**System message appears in the chat**
- Check that `system_message` is never added directly into the `Chatbot`'s `value=` list
- It should only be combined with `history` inside `bot_respond`, right before calling the API — never sent to the `Chatbot` component for display

**Persona isn't consistent**
- Make your system message more specific and detailed
- Include tone, approach, and behavior guidelines
- Try different phrasing to get better results

**Syntax errors**
- Make sure `system_message` and `initial_greeting` are each a dictionary with `"role"` and `"content"` keys
- Use proper quote matching

**Greeting doesn't match persona**
- The assistant's first message should sound like your persona
- If you're making a travel planner, say "Where would you like to go?" not "How can I help?"

## Going Further (Optional)

Want to experiment more? Try these ideas:

1. **Multiple personas with a dropdown:**
```python
persona = gr.Dropdown(["Travel Planner", "Math Tutor", "Chef"], label="Choose a persona")
# Then set system_message based on the selected value
```

2. **User-customizable system prompt:**
```python
custom_prompt = gr.Textbox(label="Define your chatbot's role", lines=3)
# Use this in your system_message
```

3. **Context-rich system messages:**
- Add specific knowledge or constraints
- "You are a travel planner specializing in budget trips to Europe"
- "You are a math tutor who only teaches algebra"

## What You've Learned

Excellent work! You've discovered one of the most powerful customization techniques in AI applications: personalization.

**Key takeaways:**
- System messages define AI behavior and personality
- They're instructions, not conversation — which is why we keep them out of `history` entirely, rather than displaying and then filtering them
- A well-crafted system prompt can transform the user experience
- This same technique powers many successful AI products

**Fun fact:** Many startups are built around this exact concept: a carefully crafted system prompt with a polished UI around the OpenAI API. You now understand the core of their technology.

In upcoming projects, you'll continue building on this foundation with even more advanced features!